# cracks-yolo demo

End-to-end walk-through of the `cracks_yolo` library for **tongue surface crack detection** (舌面裂纹检测).

This notebook covers:
1. List the 26 ZOO entries.
2. Instantiate a model and inspect its parameters.
3. Forward pass on a dummy batch.
4. Compute loss + backward.
5. Decode predictions to final detections.
6. Load official COCO pretrained weights via `from_pretrained`.
7. Run model efficiency analysis (params / MACs / latency / VRAM).

Run cells top-to-bottom. No dataset required — every cell uses either a dummy batch or synthetic targets.

In [ ]:
from __future__ import annotations

import torch

from cracks_yolo.analysis import analyze_model
from cracks_yolo.zoo import ZOO
from cracks_yolo.zoo.base import DetectorModel

## 1. Available models

`ZOO` maps short names to model classes. 26 entries span YOLOv5/v7/v8/v9/v10 with SAC/TR variants plus 6 torchvision baselines.

In [ ]:
for key in ZOO:
    print(key)

## 2. Instantiate a model

Every ZOO class accepts `num_classes` (default 1 for the cracks dataset) and `input_size` (default 640). The instance satisfies the `DetectorModel` Protocol — forward, compute_loss, decode, build_optimizer, from_pretrained, plus stride/input_size/num_classes properties.

In [ ]:
model = ZOO["yolov5s_sactr"](num_classes=1, input_size=640)
detector: DetectorModel = model  # type: ignore[assignment]

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"class:              {type(model).__name__}")
print(f"parameters:         {n_params:,}")
print(f"trainable:          {n_trainable:,}")
print(f"stride:             {detector.stride.tolist()}")
print(f"input_size:         {detector.input_size}")
print(f"num_classes:        {detector.num_classes}")
print(f"loss_parts_schema:  {detector.loss_parts_schema}")
print(f"decode_format:      {detector.decode_format}")

## 3. Forward pass

Train-mode forward returns raw predictions (anchor-based v5/v7 return a `(decoded, raw)` tuple; anchor-free v8/v9/v10 return a tensor). Eval-mode forward returns decoded predictions.

In [ ]:
torch.manual_seed(0)
model.train()
x = torch.randn(2, 3, 640, 640)
preds = model(x)

# Anchor-based (v5/v7): forward returns (decoded, raw). Anchor-free (v8+): returns a tensor.
if isinstance(preds, tuple):
    decoded, raw = preds
    print(f"decoded shape: {tuple(decoded.shape)}")
    print(f"raw shape:     {tuple(raw.shape)}")
else:
    print(f"preds shape:   {tuple(preds.shape)}")

## 4. Compute loss + backward

Targets are a `(N, 6)` tensor: `(image_idx, class_id, x_center, y_center, width, height)` — all normalized to `[0, 1]`. `compute_loss` returns `(total_loss, parts_tensor)`. YOLOv7 needs the image batch passed via `imgs=` for OTA assignment.

In [ ]:
targets = torch.tensor(
    [
        [0, 0, 0.50, 0.50, 0.20, 0.20],
        [0, 0, 0.30, 0.70, 0.10, 0.10],
        [1, 0, 0.40, 0.40, 0.15, 0.25],
    ],
    dtype=torch.float32,
)

loss, parts = detector.compute_loss(preds, targets, imgs=x)
print(f"loss:      {float(loss):.4f}")
print(f"parts:     {parts.detach().tolist()}")
print(f"schema:    {detector.loss_parts_schema}")

loss.backward()
n_with_grad = sum(1 for p in model.parameters() if p.requires_grad and p.grad is not None)
n_trainable = sum(1 for p in model.parameters() if p.requires_grad)
print(f"params with grad: {n_with_grad}/{n_trainable}")

## 5. Decode predictions

Eval-mode forward + `decode` produces a final tensor ready for NMS. Anchor-based output: `(B, N, nc+5)`. Anchor-free output: `(B, 4+nc, N)`.

In [ ]:
model.eval()
with torch.no_grad():
    out = model(x)
    decoded = detector.decode(out) if not isinstance(out, torch.Tensor) else out

print(f"decoded shape: {tuple(decoded.shape)}")
print(f"decode_format: {detector.decode_format}")
print()
print("First 5 rows of image 0 (x, y, w, h, conf, class_prob):")
print(decoded[0, :5])

## 6. Load COCO pretrained weights

`from_pretrained` downloads the official COCO checkpoint to `weights/`, then loads with `strict=False`. SAC/TR layers have no COCO weights — they appear in `LoadReport.missing` and stay randomly initialized. Baseline variants (no SAC/TR) load all keys.

In [ ]:
from pathlib import Path

from cracks_yolo.zoo import YOLOv5s

# Baseline: all COCO keys match.
model_baseline = YOLOv5s.from_pretrained(
    num_classes=1,
    weights_dir=Path("weights"),
    strict=False,
)
print(f"baseline loaded: {type(model_baseline).__name__}")

# SAC/TR variant: SAC/TR layers are missing from the COCO checkpoint.
# The loader logs a PretrainedLoadLog with the missing keys.
model_sactr = ZOO["yolov5s_sactr"].from_pretrained(
    num_classes=1,
    weights_dir=Path("weights"),
    strict=False,
)
print(f"SACTR loaded: {type(model_sactr).__name__}")

## 7. Model efficiency analysis

`analyze_model` reports parameter count, MACs (via `fvcore.FlopCountAnalysis`), inference latency (p50/p95/mean over 20 runs), and peak VRAM during forward. On CPU, latency is still measured; VRAM stays 0.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
report = analyze_model(
    ZOO["yolov5s_sactr"](num_classes=1),
    input_size=640,
    device=device,
    n_runs=20,
)

print(f"model:             {report.model_name}")
print(f"parameters:        {report.n_parameters:,}")
print(f"trainable:         {report.n_trainable_parameters:,}")
print(f"MACs:              {report.macs / 1e9:.2f} GMACs")
print(f"FLOPs:             {report.flops / 1e9:.2f} GFLOPs")
print(f"latency p50:       {report.latency_p50_ms:.2f} ms")
print(f"latency p95:       {report.latency_p95_ms:.2f} ms")
print(f"latency mean:      {report.latency_mean_ms:.2f} ms")
if report.peak_vram_bytes:
    print(f"peak VRAM:         {report.peak_vram_bytes / 1e9:.2f} GB")
print(f"device:            {report.device}")

## Next steps

- **Train**: `python -m scripts.train --model yolov5s_sactr --pretrained --dataset data/CrackDetection_Augmentation.v1.yolov5pytorch --epochs 100 --output-dir output/yolov5s_sactr`
- **5-fold CV**: add `--cross-val --n-folds 5 --val-fraction 0.1`.
- **Batch sweep**: `python -m scripts.schedule_experiments --config experiments/all_models_direct.yaml --output-dir output/all_models_direct`.
- **Full docs**: see `docs/` (English) and `docs/*.zh-CN.md` (中文).